# Mastercard Challenge 2026 — Análise Exploratória
## Priceless Bank | Diagnóstico de Perda de Market Share

**Contexto:** O Priceless Bank caiu de **33% para 19%** de market share em 4 trimestres de 2025.
O time de Advisors da Mastercard foi contratado para diagnosticar os pontos críticos e apresentar recomendações.

**Bases disponíveis:** Clientes · Cartões · Transações · PIX · Investimentos

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('Bibliotecas carregadas com sucesso.')

In [ ]:
# CORRECAO CRITICA: notebook original carregava Base_clientes.csv para base_investimentos
base_cartoes       = pd.read_csv('data/Base_cartoes.csv')
base_clientes      = pd.read_csv('data/Base_clientes.csv')
base_investimentos = pd.read_csv('data/Base_investimentos.csv')  # FIX: era Base_clientes.csv
base_pix           = pd.read_csv('data/Base_pix.csv')
base_transacoes    = pd.read_csv('data/Base_transacoes.csv')

base_cartoes['Data_Emissao']        = pd.to_datetime(base_cartoes['Data_Emissao'])
base_cartoes['Data_Ativacao']       = pd.to_datetime(base_cartoes['Data_Ativacao'])
base_cartoes['Data_Validade']       = pd.to_datetime(base_cartoes['Data_Validade'], errors='coerce')
base_clientes['Data_Nascimento']    = pd.to_datetime(base_clientes['Data_Nascimento'], dayfirst=True)
base_clientes['Data_Criacao_Conta'] = pd.to_datetime(base_clientes['Data_Criacao_Conta'])
base_transacoes['Data']             = pd.to_datetime(base_transacoes['Data'])
base_pix['Data']                    = pd.to_datetime(base_pix['Data'], errors='coerce')

print('Bases carregadas:')
for nome, df in [('Clientes', base_clientes), ('Cartoes', base_cartoes),
                 ('Transacoes', base_transacoes), ('PIX', base_pix), ('Investimentos', base_investimentos)]:
    print(f'  {nome:15s}: {df.shape[0]:>7,} linhas x {df.shape[1]} colunas')

---
## 1. Visão Geral das Bases

In [ ]:
bases = {'Clientes': base_clientes, 'Cartoes': base_cartoes,
         'Transacoes': base_transacoes, 'PIX': base_pix, 'Investimentos': base_investimentos}

resumo = []
for nome, df in bases.items():
    nulos    = df.isnull().sum().sum()
    pct_nulo = nulos / (df.shape[0] * df.shape[1]) * 100
    resumo.append({'Base': nome, 'Linhas': f'{df.shape[0]:,}', 'Colunas': df.shape[1],
                   'Nulos': f'{nulos:,}', '% Nulos': f'{pct_nulo:.1f}%',
                   'Chave': 'Cliente_ID' if 'Cliente_ID' in df.columns else 'ID proprio'})

pd.DataFrame(resumo).set_index('Base')

---
## 2. Schema Visual — Relacionamento Entre as Bases

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')
ax.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#1a1a2e')
ORANGE='#eb5e28'; TEAL='#48cae4'; GREEN='#52b788'; YELLOW='#ffd166'; WHITE='#f0f0f0'; GRAY='#888888'

def draw_entity(ax, x, y, title, fields, color, width=3.2, row_h=0.38):
    h = 0.52 + len(fields) * row_h
    ax.add_patch(mpatches.FancyBboxPatch((x-width/2, y-h), width, h,
        boxstyle='round,pad=0.05', linewidth=2, edgecolor=color, facecolor='#16213e'))
    ax.add_patch(mpatches.FancyBboxPatch((x-width/2, y-0.5), width, 0.5,
        boxstyle='round,pad=0.05', linewidth=0, facecolor=color))
    ax.text(x, y-0.25, title, color='white', fontsize=10, fontweight='bold', ha='center', va='center')
    for i,(field,ftype) in enumerate(fields):
        yf = y - 0.65 - i*row_h
        ax.text(x-width/2+0.15, yf, field, color=WHITE, fontsize=7.5, va='center')
        ax.text(x+width/2-0.1,  yf, ftype, color=GRAY,  fontsize=7,   va='center', ha='right')

def draw_arrow(ax, x1, y1, x2, y2, label='', color='#aaaaaa'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1), arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    if label: ax.text((x1+x2)/2+0.1, (y1+y2)/2, label, color=color, fontsize=7, va='center')

draw_entity(ax, 8,    9.2, 'BASE CLIENTES (1.960)',
    [('🔑 Cliente_ID','INT'),('Data_Nascimento','DATE'),('Renda_Anual','FLOAT ⚠️255 nulls'),
     ('Data_Criacao_Conta','DATE'),('Numero_Cartoes','INT'),('Cidade / Estado','STR'),
     ('Possui_Conta_Adicional','STR')], ORANGE, width=4.0)

draw_entity(ax, 2.5,  5.5, 'BASE CARTOES (4.006)',
    [('🔑 ID_Cartao','INT'),('Produto_Mastercard','STR'),('Tipo_Cartao','STR'),
     ('Data_Emissao/Ativacao','DATE ⚠️450 erros'),('Data_Validade','DATE ⚠️450 nulls'),
     ('Limite_Cartao','FLOAT')], TEAL, width=3.8)

draw_entity(ax, 8,    5.0, 'BASE TRANSACOES (156.826)',
    [('🔑 ID_Transacao','INT'),('FK Cliente_ID','INT'),('FK ID_Cartao','INT'),('Data','DATETIME'),
     ('Valor_Compra','FLOAT ⚠️165 neg.'),('Industria / Tipo_Compra','STR'),
     ('Input_Mode / Wallet','STR'),('Crossborder/Contactless','INT ⚠️95%/82% null')], YELLOW, width=4.2)

draw_entity(ax, 13.5, 5.5, 'BASE PIX (278.940)',
    [('FK Cliente_ID','INT'),('Valor','FLOAT ⚠️917 neg.'),('Data','DATETIME ⚠️418 nulls'),
     ('Pix_para_si_mesmo','INT'),('Tipo_transacao','STR'),
     ('Aprovado','INT'),('PF_PJ / Agendado','STR')], GREEN, width=3.8)

draw_entity(ax, 4.5,  1.3, 'BASE INVESTIMENTOS (21.200)',
    [('FK Cliente_ID','INT'),('Data_Abertura_Conta_Inv','INT YYYYMM'),('Data','INT YYYYMM'),
     ('Valor_Aplicado','FLOAT ⚠️1566 neg.'),('Saldo_Atual','FLOAT'),
     ('Produto_Investimento','STR'),('Data_de_vencimento','INT ⚠️299901=sem vcto')], '#a78bfa', width=4.2)

draw_arrow(ax,  8,   7.75,  8,   5.52, 'Cliente_ID (1:N)', ORANGE)
draw_arrow(ax,  6.0, 7.75,  2.5, 6.05, 'Cliente_ID (1:N)', ORANGE)
draw_arrow(ax, 10.0, 7.75, 13.5, 6.05, 'Cliente_ID (1:N)', ORANGE)
draw_arrow(ax,  6.2, 7.75,  4.5, 1.8,  'Cliente_ID (1:N)', ORANGE)
draw_arrow(ax,  4.1, 4.05,  8.0, 4.05, 'ID_Cartao (1:N)',  TEAL)

ax.text(8, 9.75, 'PRICELESS BANK — MAPA DE DADOS', color=WHITE, fontsize=13, fontweight='bold', ha='center')
ax.text(8, 9.5,  'Chave central: Cliente_ID  |  ⚠️ = anomalia detectada', color=GRAY, fontsize=9, ha='center')
plt.tight_layout()
plt.savefig('docs/schema_banco_dados.png', dpi=130, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('Schema salvo em docs/schema_banco_dados.png')

---
## 3. Cobertura dos Clientes por Base

In [ ]:
total = base_clientes['Cliente_ID'].nunique()
s_tr  = set(base_transacoes['Cliente_ID'])
s_px  = set(base_pix['Cliente_ID'])
s_inv = set(base_investimentos['Cliente_ID'])
s_cli = set(base_clientes['Cliente_ID'])

cobertura = {'Com Transacoes': len(s_tr), 'Com PIX': len(s_px),
             'Com Investimentos': len(s_inv),
             'Trans + PIX': len(s_tr & s_px), 'Trans + PIX + Inv': len(s_tr & s_px & s_inv)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cores = ['#eb5e28','#52b788','#a78bfa','#48cae4','#ffd166']
bars  = axes[0].barh(list(cobertura.keys()), list(cobertura.values()), color=cores, edgecolor='white')
axes[0].set_xlim(0, total * 1.12)
axes[0].axvline(total, color='gray', linestyle='--', label=f'Total: {total}')
for bar, v in zip(bars, cobertura.values()):
    axes[0].text(v+15, bar.get_y()+bar.get_height()/2, f'{v:,} ({v/total*100:.0f}%)', va='center', fontsize=9)
axes[0].set_title('Cobertura de Clientes por Base', fontweight='bold'); axes[0].legend()

sem_nada = len(s_cli - s_tr - s_px - s_inv)
vals_pie = [len(s_tr-s_px-s_inv), len(s_px-s_tr-s_inv), len((s_tr&s_px)-s_inv), len(s_tr&s_px&s_inv), sem_nada]
labs_pie = ['So Trans.','So PIX','Trans+PIX','Trans+PIX+Inv','Sem atividade']
axes[1].pie(vals_pie, labels=labs_pie, autopct='%1.1f%%', startangle=140,
            colors=['#ffd166','#52b788','#48cae4','#eb5e28','#888888'],
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Distribuicao dos 1.960 Clientes por Atividade', fontweight='bold')

plt.tight_layout(); plt.show()
print(f'Clientes sem NENHUMA atividade: {sem_nada}')

---
## 4. Base Clientes — Perfil Demográfico e Financeiro

In [ ]:
hoje = pd.Timestamp('2026-06-12')
base_clientes['Idade'] = ((hoje - base_clientes['Data_Nascimento']).dt.days / 365.25).astype(int)
base_clientes['Faixa_Etaria'] = pd.cut(base_clientes['Idade'],
    bins=[18,30,40,50,60,70,80], labels=['18-29','30-39','40-49','50-59','60-69','70+'], right=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Base Clientes — Perfil Completo', fontsize=14, fontweight='bold')

axes[0,0].hist(base_clientes['Idade'], bins=20, color='#eb5e28', edgecolor='white', linewidth=0.5)
axes[0,0].axvline(base_clientes['Idade'].mean(), color='black', linestyle='--',
                  label=f'Media: {base_clientes["Idade"].mean():.0f} anos')
axes[0,0].set_title('Distribuicao de Idade'); axes[0,0].legend(fontsize=8)

fc = base_clientes['Faixa_Etaria'].value_counts().sort_index()
axes[0,1].bar(fc.index, fc.values, color='#48cae4', edgecolor='white')
axes[0,1].set_title('Clientes por Faixa Etaria')
for i,v in enumerate(fc.values): axes[0,1].text(i, v+5, str(v), ha='center', fontsize=8)

renda = base_clientes['Renda_Anual'].dropna()
axes[0,2].hist(renda/1000, bins=20, color='#52b788', edgecolor='white', linewidth=0.5)
axes[0,2].axvline(renda.mean()/1000, color='black', linestyle='--', label=f'Media: R${renda.mean()/1000:.0f}k')
axes[0,2].set_title(f'Renda Anual (255 nulls = {255/1960*100:.1f}%)'); axes[0,2].legend(fontsize=8)
axes[0,2].set_xlabel('R$ mil')

ec = base_clientes['Estado'].value_counts()
axes[1,0].bar(ec.index, ec.values, color='#ffd166', edgecolor='white')
axes[1,0].set_title('Clientes por Estado'); axes[1,0].tick_params(axis='x', rotation=45)

nc = base_clientes['Numero_Cartoes'].value_counts().sort_index()
axes[1,1].bar(nc.index.astype(str), nc.values, color='#a78bfa', edgecolor='white')
axes[1,1].set_title('Nr de Cartoes por Cliente')
for i,v in enumerate(nc.values): axes[1,1].text(i, v+5, str(v), ha='center', fontsize=9)

ca = base_clientes['Possui_Conta_Adicional'].value_counts()
axes[1,2].pie(ca.values, labels=ca.index, autopct='%1.1f%%',
              colors=['#52b788','#eb5e28'], wedgeprops={'edgecolor':'white'})
axes[1,2].set_title('Possui Conta Adicional')

plt.tight_layout(); plt.show()
print(f'Idade media: {base_clientes["Idade"].mean():.1f} | Mediana: {base_clientes["Idade"].median():.0f}')
print(f'Renda media: R${renda.mean():,.0f} | Mediana: R${renda.median():,.0f}')

---
## 5. Base Cartões — Portfolio e Anomalias

In [ ]:
hoje = pd.Timestamp('2026-06-12')
ativ_antes = base_cartoes[base_cartoes['Data_Ativacao'] < base_cartoes['Data_Emissao']]
sem_val    = base_cartoes[base_cartoes['Data_Validade'].isnull()]
vencidos   = base_cartoes[base_cartoes['Data_Validade'].notna() & (base_cartoes['Data_Validade'] < hoje)]
inativos   = set(base_cartoes['ID_Cartao']) - set(base_transacoes['ID_Cartao'])

print('=== ANOMALIAS ===')
print(f'[!] Ativados ANTES da emissao: {len(ativ_antes)} ({len(ativ_antes)/len(base_cartoes)*100:.1f}%)')
print(f'    Sao EXATAMENTE os mesmos sem Data_Validade: {len(set(ativ_antes["ID_Cartao"]) & set(sem_val["ID_Cartao"])) == 450}')
print(f'[!] Cartoes vencidos: {len(vencidos)} de {base_cartoes["Data_Validade"].notna().sum()}')
print(f'[!] Cartoes sem nenhuma transacao: {len(inativos)} ({len(inativos)/len(base_cartoes)*100:.1f}%) — baixo engajamento')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Base Cartoes — Portfolio Mastercard', fontsize=13, fontweight='bold')

prod  = base_cartoes['Produto_Mastercard'].value_counts()
cores = ['#ffd166','#eb5e28','#48cae4','#52b788','#a78bfa']
bars  = axes[0].bar(prod.index, prod.values, color=cores, edgecolor='white')
axes[0].set_title('Cartoes por Produto'); axes[0].tick_params(axis='x', rotation=30)
for bar,v in zip(bars,prod.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+10, f'{v}\n({v/len(base_cartoes)*100:.0f}%)', ha='center', fontsize=8)

credito    = base_cartoes[base_cartoes['Limite_Cartao'] > 0]
ordem      = ['Standard','Gold','Platinum','Black']
dados_box  = [credito[credito['Produto_Mastercard']==p]['Limite_Cartao'].values for p in ordem]
bp = axes[1].boxplot(dados_box, labels=ordem, patch_artist=True, medianprops={'color':'black','linewidth':2})
for patch,cor in zip(bp['boxes'],['#48cae4','#ffd166','#eb5e28','#a78bfa']):
    patch.set_facecolor(cor); patch.set_alpha(0.8)
axes[1].set_title('Limite por Produto (R$)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'R${x/1000:.0f}k'))

base_cartoes['Mes_Emissao'] = base_cartoes['Data_Emissao'].dt.to_period('M')
em = base_cartoes.groupby('Mes_Emissao').size()
axes[2].plot(em.index.astype(str), em.values, color='#eb5e28', linewidth=2, marker='o', markersize=4)
axes[2].set_title('Emissao de Cartoes por Mes'); axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()
print(f'Limite medio: R${credito["Limite_Cartao"].mean():,.0f} | Mediano: R${credito["Limite_Cartao"].median():,.0f}')

---
## 6. Base Transações — Comportamento de Consumo
### ⭐ Base mais crítica para o diagnóstico

In [ ]:
print('=== ANOMALIAS ===')
print(f'[!] Crossborder NULL: {base_transacoes["Crossborder"].isnull().mean()*100:.1f}% — por design: so em CNP online')
print(f'[!] Contactless NULL: {base_transacoes["Contactless"].isnull().mean()*100:.1f}% — por design: so em fisico')
print(f'[!] Valores negativos (estornos?): {(base_transacoes["Valor_Compra"]<0).sum()}')
print(f'[!] Outliers > R$10k: {(base_transacoes["Valor_Compra"]>10000).sum()} (max: R${base_transacoes["Valor_Compra"].max():,.0f})')

tr = base_transacoes[base_transacoes['Valor_Compra'] > 0]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Base Transacoes — Analise Completa', fontsize=13, fontweight='bold')

ind   = tr['Industria'].value_counts()
cores = ['#eb5e28','#48cae4','#ffd166','#52b788','#a78bfa','#f77f00']
bars  = axes[0,0].barh(ind.index, ind.values, color=cores, edgecolor='white')
axes[0,0].set_title('Transacoes por Industria')
for bar,v in zip(bars,ind.values):
    axes[0,0].text(v+200, bar.get_y()+bar.get_height()/2, f'{v:,}', va='center', fontsize=8)

vm = tr.groupby('Industria')['Valor_Compra'].mean().sort_values(ascending=False)
b2 = axes[0,1].bar(vm.index, vm.values, color='#48cae4', edgecolor='white')
axes[0,1].set_title('Ticket Medio por Industria (R$)'); axes[0,1].tick_params(axis='x', rotation=45)
for bar,v in zip(b2,vm.values):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, v+5, f'R${v:,.0f}', ha='center', fontsize=7, rotation=45)

im = tr['Input_Mode'].value_counts()
axes[0,2].pie(im.values, labels=im.index, autopct='%1.1f%%', startangle=90,
              colors=['#eb5e28','#48cae4','#52b788','#ffd166','#a78bfa','#f77f00','#e63946','#457b9d'],
              wedgeprops={'edgecolor':'white'})
axes[0,2].set_title('Canal de Entrada (Input Mode)')

tc = tr['Tipo_Compra'].value_counts()
axes[1,0].pie(tc.values, labels=['CP (Presencial)','CNP (Online)'],
              autopct='%1.1f%%', colors=['#52b788','#ffd166'], wedgeprops={'edgecolor':'white'})
axes[1,0].set_title('Presencial (CP) vs Online (CNP)')

tso = tr[tr['Valor_Compra'] < 5000]
axes[1,1].hist(tso['Valor_Compra'], bins=50, color='#eb5e28', edgecolor='white', linewidth=0.3)
axes[1,1].axvline(tso['Valor_Compra'].median(), color='black', linestyle='--',
                  label=f'Mediana: R${tso["Valor_Compra"].median():.0f}')
axes[1,1].set_title('Distribuicao de Valores (< R$5k)'); axes[1,1].legend(fontsize=8)

parc = tr['Qtd_Parcelas'].dropna()
axes[1,2].hist(parc, bins=12, range=(1,13), color='#52b788', edgecolor='white', linewidth=0.5)
axes[1,2].set_title(f'Parcelas ({len(parc):,} transacoes = {len(parc)/len(tr)*100:.1f}%)')

plt.tight_layout(); plt.show()
print(f'Ticket medio: R${tr["Valor_Compra"].mean():,.0f} | Mediana: R${tr["Valor_Compra"].median():,.0f}')
print(f'Top industria: {ind.index[0]} ({ind.iloc[0]:,}) | Maior ticket: {vm.index[0]} (R${vm.iloc[0]:,.0f})')

---
## 7. Base PIX — Comportamento de Transferências
### ⭐ Segunda base mais crítica — evidência de canibalização do cartão

In [ ]:
env_naprov = len(base_pix[(base_pix['Tipo_transacao']=='Envio') & (base_pix['Aprovado']==0)])
tot_env    = base_pix['Tipo_transacao'].value_counts().get('Envio', 1)
print('=== ANOMALIAS ===')
print(f'[!] Valores negativos: {(base_pix["Valor"]<0).sum()} — PIX negativo e incoerente')
print(f'[!] Valores zero: {(base_pix["Valor"]==0).sum()} — PIX de R$0,00')
print(f'[!] Data/Tipo nulos: {base_pix["Data"].isnull().sum()} registros incompletos')
print(f'[!] Recebimentos negados (Aprovado=0): {len(base_pix[(base_pix["Tipo_transacao"]=="Recebimento") & (base_pix["Aprovado"]==0)]):,}')
print(f'[!] Envios nao aprovados: {env_naprov:,} ({env_naprov/tot_env*100:.1f}%) — taxa alta')

pix_ok = base_pix[base_pix['Valor'] > 0].dropna(subset=['Data','Tipo_transacao'])

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Base PIX — Analise Completa (278.940 registros)', fontsize=13, fontweight='bold')

tt = base_pix['Tipo_transacao'].value_counts()
axes[0,0].pie(tt.values, labels=tt.index, autopct='%1.1f%%',
              colors=['#eb5e28','#48cae4'], wedgeprops={'edgecolor':'white'})
axes[0,0].set_title('Envio vs Recebimento')

pfpj = base_pix['PF_PJ'].value_counts()
axes[0,1].pie(pfpj.values, labels=pfpj.index, autopct='%1.1f%%',
              colors=['#52b788','#ffd166'], wedgeprops={'edgecolor':'white'})
axes[0,1].set_title('Destinatario PF vs PJ')

aprov = base_pix.groupby('Tipo_transacao')['Aprovado'].mean() * 100
axes[0,2].bar(aprov.index, aprov.values, color=['#eb5e28','#48cae4'], edgecolor='white')
axes[0,2].set_ylim(0, 110); axes[0,2].set_title('Taxa de Aprovacao por Tipo (%)')
for i,v in enumerate(aprov.values): axes[0,2].text(i, v+1, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

pso = pix_ok[pix_ok['Valor'] < 2000]
axes[1,0].hist(pso['Valor'], bins=40, color='#52b788', edgecolor='white', linewidth=0.3)
axes[1,0].axvline(pso['Valor'].median(), color='black', linestyle='--',
                  label=f'Mediana: R${pso["Valor"].median():.0f}')
axes[1,0].set_title('Distribuicao de Valores PIX (< R$2k)'); axes[1,0].legend(fontsize=8)

psm = base_pix['Pix_para_si_mesmo'].value_counts()
axes[1,1].pie(psm.values, labels=['Para terceiros','Para si mesmo'],
              autopct='%1.1f%%', colors=['#a78bfa','#ffd166'], wedgeprops={'edgecolor':'white'})
axes[1,1].set_title('PIX para Si Mesmo vs Terceiros')

ag = base_pix['Agendado'].value_counts()
axes[1,2].bar(ag.index, ag.values, color=['#48cae4','#eb5e28'], edgecolor='white')
axes[1,2].set_title('PIX Agendado vs Imediato')
for i,v in enumerate(ag.values): axes[1,2].text(i, v+500, f'{v:,}\n({v/len(base_pix)*100:.1f}%)', ha='center', fontsize=9)

plt.tight_layout(); plt.show()
print(f'Valor medio PIX (positivos): R${pix_ok["Valor"].mean():,.0f} | Mediana: R${pix_ok["Valor"].median():,.0f}')
print('62% dos PIX sao para PJ: clientes pagando em comercios via PIX em vez de cartao')

---
## 8. Base Investimentos — Carteira e Comportamento

In [ ]:
print('=== NOTAS (comportamento esperado, nao sao erros) ===')
print(f'[i] Valor_Aplicado negativo: {(base_investimentos["Valor_Aplicado"]<0).sum():,} — sao RESGATES')
print(f'[i] Data_vencimento=299901: {(base_investimentos["Data_de_vencimento"]==299901).sum():,} — Reservinha sem vencimento (esperado)')
print(f'[i] Saldo_Atual=0: {(base_investimentos["Saldo_Atual"]==0).sum():,} — produto vencido ou resgatado')
print(f'[i] Datas em formato YYYYMM inteiro: {base_investimentos["Data"].dtype}')
print(f'[i] Clientes com investimento: {base_investimentos["Cliente_ID"].nunique()} ({base_investimentos["Cliente_ID"].nunique()/1960*100:.1f}% da base)')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Base Investimentos — Carteira do Priceless Bank', fontsize=13, fontweight='bold')

pi = base_investimentos['Produto_Investimento'].value_counts()
axes[0].pie(pi.values, labels=pi.index, autopct='%1.1f%%',
            colors=['#48cae4','#ffd166','#52b788','#a78bfa'], wedgeprops={'edgecolor':'white'})
axes[0].set_title('Distribuicao por Produto')

prods = base_investimentos['Produto_Investimento'].unique()
ds    = [base_investimentos[base_investimentos['Produto_Investimento']==p]['Saldo_Atual'].values for p in prods]
bp = axes[1].boxplot(ds, labels=prods, patch_artist=True, medianprops={'color':'black','linewidth':2})
for patch,cor in zip(bp['boxes'],['#48cae4','#ffd166','#52b788','#a78bfa']):
    patch.set_facecolor(cor); patch.set_alpha(0.8)
axes[1].set_title('Saldo por Produto (R$)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'R${x/1000:.0f}k'))
axes[1].tick_params(axis='x', rotation=15)

im = base_investimentos[base_investimentos['Valor_Aplicado'] > 0].copy()
im['ds'] = im['Data'].astype(str)
im = im[im['ds'].str[:4].astype(int).between(2023,2025) & im['ds'].str[4:].astype(int).between(1,12)]
im['Period'] = im['ds'].apply(lambda x: f'{x[:4]}-{x[4:]}')
ap = im.groupby('Period')['Valor_Aplicado'].sum().sort_index()
axes[2].bar(range(len(ap)), ap.values/1000, color='#52b788', edgecolor='white')
step = max(1, len(ap)//8)
axes[2].set_xticks(range(0,len(ap),step))
axes[2].set_xticklabels(ap.index[::step], rotation=45, fontsize=7)
axes[2].set_title('Aportes Mensais (R$ mil)')

plt.tight_layout(); plt.show()

sp = base_investimentos.groupby('Produto_Investimento')['Saldo_Atual'].agg(['mean','sum','count'])
sp.columns = ['Saldo Medio','Saldo Total','Registros']
sp['Saldo Total'] = sp['Saldo Total'].apply(lambda x: f'R${x/1e6:.1f}M')
sp['Saldo Medio'] = sp['Saldo Medio'].apply(lambda x: f'R${x:,.0f}')
print('\nSaldo por produto:'); print(sp)

---
## 9. Análise Temporal — O Sinal Mais Crítico do Diagnóstico

Evolução trimestral do volume transacionado por cartão vs PIX.

In [ ]:
base_transacoes['YQ'] = base_transacoes['Data'].dt.to_period('Q')
pix2 = base_pix[base_pix['Valor'] > 0].dropna(subset=['Data']).copy()
pix2['YQ'] = pix2['Data'].dt.to_period('Q')

anos = ['2023','2024','2025']
tq = base_transacoes[base_transacoes['Valor_Compra']>0].groupby('YQ').agg(
    Volume=('Valor_Compra','sum'), Qtd=('Valor_Compra','count')).reset_index()
tq = tq[tq['YQ'].astype(str).str[:4].isin(anos)]

pq = pix2[pix2['Tipo_transacao']=='Envio'].groupby('YQ').agg(
    Volume=('Valor','sum'), Qtd=('Valor','count')).reset_index()
pq = pq[pq['YQ'].astype(str).str[:4].isin(anos)]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('ANALISE TEMPORAL — Cartao vs PIX\n(Evidencia Central da Perda de Market Share)',
             fontsize=13, fontweight='bold', color='#eb5e28')

xs = range(len(tq))
bt = axes[0,0].bar(xs, tq['Volume']/1e6, color='#ffd166', edgecolor='white')
axes[0,0].set_title('Volume — Cartao (R$ milhoes)', fontweight='bold')
axes[0,0].set_xticks(xs); axes[0,0].set_xticklabels(tq['YQ'].astype(str), rotation=45, fontsize=8)
for bar,v in zip(bt,tq['Volume']):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'R${v/1e6:.1f}M', ha='center', fontsize=7, rotation=90)

xs2 = range(len(pq))
bp2 = axes[0,1].bar(xs2, pq['Volume']/1e6, color='#52b788', edgecolor='white')
axes[0,1].set_title('Volume — PIX Enviado (R$ milhoes)', fontweight='bold')
axes[0,1].set_xticks(xs2); axes[0,1].set_xticklabels(pq['YQ'].astype(str), rotation=45, fontsize=8)
for bar,v in zip(bp2,pq['Volume']):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'R${v/1e6:.1f}M', ha='center', fontsize=7, rotation=90)

all_p  = sorted(set(tq['YQ'].astype(str)) | set(pq['YQ'].astype(str)))
tqd    = dict(zip(tq['YQ'].astype(str), tq['Qtd']))
pqd    = dict(zip(pq['YQ'].astype(str), pq['Qtd']))
x = np.arange(len(all_p)); w = 0.4
axes[1,0].bar(x-w/2, [tqd.get(p,0) for p in all_p], w, label='Cartao', color='#ffd166', edgecolor='white')
axes[1,0].bar(x+w/2, [pqd.get(p,0) for p in all_p], w, label='PIX (envios)', color='#52b788', edgecolor='white')
axes[1,0].set_title('Quantidade: Cartao vs PIX', fontweight='bold')
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(all_p, rotation=45, fontsize=7); axes[1,0].legend()

pc   = [p for p in all_p if p in tqd and p in pqd]
razao = [pqd[p]/tqd[p] for p in pc]
axes[1,1].plot(pc, razao, color='#eb5e28', linewidth=2.5, marker='o', markersize=8)
axes[1,1].fill_between(pc, razao, alpha=0.2, color='#eb5e28')
axes[1,1].axhline(1.0, color='gray', linestyle='--', label='PIX = Cartao')
axes[1,1].set_title('Razao PIX/Cartao (> 1 = PIX supera cartao)', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=45); axes[1,1].legend(fontsize=9)
for p,r in zip(pc,razao): axes[1,1].annotate(f'{r:.1f}x',(p,r), xytext=(5,5), textcoords='offset points', fontsize=8)

plt.tight_layout()
plt.savefig('docs/analise_temporal_cartao_pix.png', dpi=130, bbox_inches='tight')
plt.show()

v_c24 = tq[tq['YQ'].astype(str)=='2024Q4']['Volume']
v_c25 = tq[tq['YQ'].astype(str)=='2025Q4']['Volume']
v_p24 = pq[pq['YQ'].astype(str)=='2024Q4']['Volume']
v_p25 = pq[pq['YQ'].astype(str)=='2025Q4']['Volume']
print('\n=== ACHADO CRITICO ===')
if len(v_c24) and len(v_c25):
    print(f'Cartao: R${v_c24.values[0]/1e6:.1f}M (2024Q4) -> R${v_c25.values[0]/1e6:.1f}M (2025Q4) = queda de {(1-v_c25.values[0]/v_c24.values[0])*100:.0f}%')
if len(v_p24) and len(v_p25):
    print(f'PIX:    R${v_p24.values[0]/1e6:.1f}M (2024Q4) -> R${v_p25.values[0]/1e6:.1f}M (2025Q4) = crescimento de {(v_p25.values[0]/v_p24.values[0]-1)*100:.0f}%')

---
## 10. Radar de Qualidade — Todas as Pegadinhas

In [ ]:
issues = [
    ('Notebook',      'CRITICO',  'base_investimentos carregava Base_clientes.csv',        1,     'CRITICO', 'Bug no codigo original — dado errado!',          'Corrigido neste notebook'),
    ('Cartoes',       'ERRO',     'Data_Ativacao < Data_Emissao',                        450,     'Medio',   'Bug na geracao de dados',                        'Excluir esses 450 da analise temporal'),
    ('Cartoes',       'ERRO',     'Data_Validade nula (mesmo grupo dos 450)',             450,     'Medio',   'Ligado ao problema anterior',                    'Excluir da analise temporal'),
    ('Cartoes',       'ANALISE',  'Cartoes sem nenhuma transacao',                       469,     'Alto',    '11.7% nunca usados — falha de onboarding',       'Investigar segmentos e produtos'),
    ('Clientes',      'NULO',     'Renda_Anual nula',                                    255,     'Medio',   '13% sem renda declarada',                        'Imputar mediana ou flag separado'),
    ('Transacoes',    'ANALISE',  'Crossborder NULL 95.6%',                           149912,     'Baixo',   'Por design: so em CNP online',                   'Usar apenas onde nao eh nulo'),
    ('Transacoes',    'ANALISE',  'Contactless NULL 82.5%',                           129332,     'Baixo',   'Por design: so em fisico',                       'Usar apenas onde nao eh nulo'),
    ('Transacoes',    'ERRO',     'Valor_Compra negativo',                               165,     'Medio',   'Possiveis estornos/chargebacks',                  'Excluir ou criar flag estorno'),
    ('Transacoes',    'ANALISE',  'Outliers > R$10k',                                     84,     'Medio',   'Transacoes extremas distorcem media',             'Tratar separado na analise'),
    ('PIX',           'ERRO',     'Valor negativo',                                      917,     'Alto',    'PIX com valor negativo — incoerente',             'Investigar — possivelmente estornos'),
    ('PIX',           'ERRO',     'Valor zero',                                          591,     'Alto',    'PIX de R$0,00 sem sentido',                      'Excluir da analise'),
    ('PIX',           'NULO',     'Data e Tipo_transacao nulos',                         418,     'Medio',   'Registros incompletos',                           'Excluir da analise temporal'),
    ('PIX',           'SUSPEITO', 'Recebimento com Aprovado=0',                         3392,     'Alto',    'Recebimento negado — suspeito',                  'Investigar — pode ser fraude'),
    ('Investimentos', 'ANALISE',  'Valor_Aplicado negativo (resgates)',                 1566,     'Baixo',   'Comportamento normal de carteira',                'Separar aportes de resgates'),
    ('Investimentos', 'ANALISE',  'Data_vencimento=299901 (Reservinha)',                7864,     'Baixo',   'Por design — liquidez diaria sem vencimento',     'Tratar como produto especial'),
]

df_iss = pd.DataFrame(issues, columns=['Base','Tipo','Problema','Qtd','Impacto','Explicacao','Acao'])

def ci(v): return {'CRITICO':'background-color:#8b0000;color:white','Alto':'background-color:#c0392b;color:white',
                    'Medio':'background-color:#e67e22;color:white','Baixo':'background-color:#27ae60;color:white'}.get(v,'')
def ct(v): return {'CRITICO':'background-color:#8b0000;color:white','ERRO':'background-color:#c0392b;color:white',
                    'SUSPEITO':'background-color:#8e44ad;color:white','NULO':'background-color:#e67e22;color:white',
                    'ANALISE':'background-color:#2980b9;color:white'}.get(v,'')

print(f'Total: {len(df_iss)} problemas | CRITICO: {(df_iss["Impacto"]=="CRITICO").sum()} | Alto: {(df_iss["Impacto"]=="Alto").sum()} | Medio: {(df_iss["Impacto"]=="Medio").sum()} | Baixo: {(df_iss["Impacto"]=="Baixo").sum()}')
(df_iss.style.applymap(ci, subset=['Impacto']).applymap(ct, subset=['Tipo'])
  .set_properties(**{'font-size':'10px'}).hide(axis='index'))

---
## 11. Ranking das Bases — Criticidade para o Diagnóstico

In [ ]:
ranking = [
    ('1o','Transacoes',   '156.826','10/10','Volume caiu 75% de 2024Q4->2025Q4. Mostra QUEM compra, QUANTO, EM QUAL SETOR e CANAL.','Volume temporal · Ticket por industria · Canais · Parcelamento','#FFD700'),
    ('2o','PIX',          '278.940', '9/10','62% dos envios vao para PJ: clientes pagando em comercios via PIX em vez de cartao. Volume explodiu 6x em 2025.','PIX para PJ vs cartao · Crescimento 2025 · Taxa de aprovacao','#C0C0C0'),
    ('3o','Clientes',      '1.960',  '8/10','Tabela mestre. Idade media 49 anos + renda R$85k. Nao atrai o publico jovem digital-first do LuminaPay.','Segmentacao renda/idade · Correlacao renda x limite · Churn','#CD7F32'),
    ('4o','Cartoes',       '4.006',  '6/10','469 inativos (11.7%) — falha de engajamento. Util para limite vs gasto e cartoes vencidos.','Taxa de ativacao · Limite vs gasto · Cartoes vencidos','#48cae4'),
    ('5o','Investimentos','21.200',  '5/10','So 51% dos clientes. Reservinha pode ser ancora de retencao — clientes fieis tendem a investir mais.','Correlacao invest x cartao · Saldo medio · Taxa de resgate','#a78bfa'),
]

fig, ax = plt.subplots(figsize=(16, 6.5))
ax.axis('off')
for i,(rank,base,regs,score,just,analises,cor) in enumerate(ranking):
    y = 0.9 - i*0.19
    ax.add_patch(mpatches.FancyBboxPatch((0.01,y-0.065), 0.98, 0.19,
        boxstyle='round,pad=0.01', linewidth=2, edgecolor=cor, facecolor='#f8f9fa'))
    ax.text(0.03, y+0.07, rank,  fontsize=14, fontweight='bold', va='top', color=cor)
    ax.text(0.10, y+0.07, f'{base.upper()}  ({regs} registros)  —  Score: {score}', fontsize=10, fontweight='bold', va='top')
    ax.text(0.10, y+0.00, just,    fontsize=8, va='top', color='#333333')
    ax.text(0.10, y-0.045, f'Analises: {analises}', fontsize=7.5, va='top', color=cor, style='italic')
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title('RANKING DE BASES — Criticidade para o Diagnostico do Priceless Bank', fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('docs/ranking_bases.png', dpi=130, bbox_inches='tight')
plt.show()

---
## 12. Key Insights — Diagnóstico Preliminar

> Estes insights orientam as hipóteses para a apresentação final ao Comitê Executivo da Mastercard.

### 🔴 Insight 1 — Canibalização do Cartão pelo PIX [CRÍTICO]
Volume por cartão **caiu ~75%** de 2024Q4 para 2025Q4. PIX **cresceu 6x** no mesmo período. **62% dos PIX vão para PJ** — clientes pagando em estabelecimentos via PIX em vez do cartão. Hipótese: LuminaPay oferece PIX no crédito; o Priceless Bank não tem equivalente.

### 🔴 Insight 2 — 11,7% dos Cartões Nunca Foram Usados [CRÍTICO]
**469 cartões** emitidos e nunca utilizados. Falha de onboarding ou ausência de incentivo pós-emissão — forte preditor de churn futuro.

### 🟡 Insight 3 — Perfil Demográfico Desalinhado com o Mercado em Crescimento
Idade média **49 anos**, renda média **R$85k**. O LuminaPay (principal ganhador) foca em jovens adultos e early adopters — público que o Priceless Bank não está capturando.

### 🟡 Insight 4 — Taxa de Reprovação de PIX Elevada
**10% dos envios não aprovados** (24.644 registros). Taxa anormalmente alta — pode indicar fraude, limites restritivos ou fricção no processo digital.

### 🟢 Insight 5 — Reservinha como Âncora de Retenção Subutilizada
**37% dos registros de investimento** são da Reservinha (liquidez diária). Clientes com investimentos ativos tendem a ser mais fiéis — pode ser alavancada como diferencial competitivo.

---
### Próximos Passos
1. **Mapa de fugitivos:** Clientes com queda de cartão + aumento de PIX para PJ
2. **Segmento em risco:** Qual faixa etária/renda mais reduziu uso do cartão em 2025?
3. **Hipótese de fidelidade:** Clientes com investimento têm maior gasto no cartão?
4. **Cartões inativos:** Quais produtos/segmentos concentram mais não-ativações?